In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import feature_engineering
import cleaning_script

In [2]:
#loading raw datasets
feb = pd.read_csv('../original_datasets/CRMLSSold202502_filled.csv')
mar = pd.read_csv('../original_datasets/CRMLSSold202503_filled.csv')
apr = pd.read_csv('../original_datasets/CRMLSSold202504_filled.csv')
may = pd.read_csv('../original_datasets/CRMLSSold202505_filled.csv')
jun = pd.read_csv('../original_datasets/CRMLSSold202506_filled.csv')
july = pd.read_csv('../original_datasets/CRMLSSold202507_filled.csv')
aug = pd.read_csv('../original_datasets/CRMLSSold202508_filled.csv')
sep = pd.read_csv('../original_datasets/CRMLSSold202509_filled.csv')
oct = pd.read_csv('../original_datasets/CRMLSSold202510_filled.csv')

C:\Users\jueeh\AppData\Local\Temp\ipykernel_924\4101254706.py:6: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  jun = pd.read_csv('../original_datasets/CRMLSSold202506_filled.csv')


In [3]:
#build raw training and testing datasets
training_raw = pd.concat([feb, mar, apr, may, jun, july, aug, sep], ignore_index=True)
testing_raw = oct.copy()

#training_raw.to_csv('train_raw.csv', index=False)
#testing_raw.to_csv('test_raw.csv', index=False)

In [4]:
cleaned_df_train = cleaning_script.clean_real_estate_data(training_raw)
cleaned_df_train.columns

Index(['ViewYN', 'PoolPrivateYN', 'CloseDate', 'ClosePrice', 'Latitude',
       'Longitude', 'LivingArea', 'CountyOrParish', 'AttachedGarageYN',
       'ParkingTotal', 'LotSizeAcres', 'YearBuilt', 'BathroomsTotalInteger',
       'City', 'BedroomsTotal', 'ContractStatusChangeDate',
       'PurchaseContractDate', 'ListingContractDate', 'FireplaceYN', 'Stories',
       'LotSizeArea', 'NewConstructionYN', 'GarageSpaces', 'PostalCode',
       'LotSizeSquareFeet', 'PropertyAgeAtClose'],
      dtype='object')

In [5]:
cleaned_df_testing = cleaning_script.clean_real_estate_data(testing_raw)

In [6]:
cleaned_df_testing.columns

Index(['ViewYN', 'PoolPrivateYN', 'CloseDate', 'ClosePrice', 'Latitude',
       'Longitude', 'LivingArea', 'CountyOrParish', 'AttachedGarageYN',
       'ParkingTotal', 'LotSizeAcres', 'YearBuilt', 'BathroomsTotalInteger',
       'City', 'BedroomsTotal', 'ContractStatusChangeDate',
       'PurchaseContractDate', 'ListingContractDate', 'FireplaceYN', 'Stories',
       'LotSizeArea', 'NewConstructionYN', 'GarageSpaces', 'PostalCode',
       'LotSizeSquareFeet', 'PropertyAgeAtClose'],
      dtype='object')

In [7]:
train_features, test_features = feature_engineering.build_features(cleaned_df_train, cleaned_df_testing)

In [10]:
train_features.max()

ViewYN                        True
PoolPrivateYN                 True
ClosePrice               8500000.0
Latitude                 41.894714
Longitude                    329.0
LivingArea                 14168.0
AttachedGarageYN              True
ParkingTotal               15720.0
LotSizeAcres               94090.0
YearBuilt                   2026.0
BathroomsTotalInteger         45.0
BedroomsTotal                 45.0
FireplaceYN                   True
Stories                        2.0
NewConstructionYN             True
GarageSpaces               15720.0
PropertyAgeAtClose           249.0
CloseYear                     2025
CloseMonth                       9
CloseQuarter                     3
CloseDayOfWeek                   6
DaysOnMarket                  3761
DaysOfferToClose              1492
LogLivingArea             9.558812
LogLotSize               21.459099
PricePerSqFt             8500000.0
BathsPerBedroom               7.75
GarageToParking                4.0
HasPoolWithView     

In [11]:
test_x = test_features.drop(columns = 'ClosePrice')
test_y = test_features[['ClosePrice']]
train_x = train_features.drop(columns = 'ClosePrice')
train_y = train_features[['ClosePrice']]

In [12]:
print("FEATURE ALIGNMENT CHECK:")
if list(train_x.columns) == list(test_x.columns):
    print("  ✓ Train and test features match perfectly!")
else:
    print("  ⚠️  WARNING: Feature mismatch detected!")
    train_only = set(train_x.columns) - set(test_x.columns)
    test_only = set(test_x.columns) - set(train_x.columns)
    if train_only:
        print(f"    Features only in train: {train_only}")
    if test_only:
        print(f"    Features only in test: {test_only}")

FEATURE ALIGNMENT CHECK:
  ✓ Train and test features match perfectly!


In [13]:
test_x.max()

ViewYN                        True
PoolPrivateYN                 True
Latitude                 41.868066
Longitude               -11.230259
LivingArea                 14090.0
AttachedGarageYN              True
ParkingTotal                 821.0
LotSizeAcres              127631.0
YearBuilt                   2026.0
BathroomsTotalInteger         11.0
BedroomsTotal                 11.0
FireplaceYN                   True
Stories                        2.0
NewConstructionYN             True
GarageSpaces                 821.0
PropertyAgeAtClose           145.0
CloseYear                     2025
CloseMonth                      10
CloseQuarter                     4
CloseDayOfWeek                   6
DaysOnMarket                  1004
DaysOfferToClose               932
LogLivingArea             9.553292
LogLotSize               20.634172
PricePerSqFt             8000000.0
BathsPerBedroom                2.0
GarageToParking                5.0
HasPoolWithView                  1
IsSingleStory       

In [14]:
#BUILDING BASELINE XGBOOST MODEL
# Initialize XGBoost with reasonable defaults
model = xgb.XGBRegressor(
    n_estimators=100,        # Number of trees
    learning_rate=0.1,       # Step size for each tree
    max_depth=6,             # Maximum depth of each tree
    min_child_weight=1,      # Minimum samples in leaf
    subsample=0.8,           # Fraction of samples for each tree
    colsample_bytree=0.8,    # Fraction of features for each tree
    random_state=42,
    n_jobs=-1                # Use all CPU cores
)

print("Model parameters:")
for param, value in model.get_params().items():
    print(f"  {param}: {value}")

# Train the model
print("\nTraining model...")
model.fit(
    train_x, train_y,
    eval_set=[(train_x, train_y), (test_x, test_y)],
    verbose=False
)
print("Training Complete")

Model parameters:
  objective: reg:squarederror
  base_score: None
  booster: None
  callbacks: None
  colsample_bylevel: None
  colsample_bynode: None
  colsample_bytree: 0.8
  device: None
  early_stopping_rounds: None
  enable_categorical: False
  eval_metric: None
  feature_types: None
  feature_weights: None
  gamma: None
  grow_policy: None
  importance_type: None
  interaction_constraints: None
  learning_rate: 0.1
  max_bin: None
  max_cat_threshold: None
  max_cat_to_onehot: None
  max_delta_step: None
  max_depth: 6
  max_leaves: None
  min_child_weight: 1
  missing: nan
  monotone_constraints: None
  multi_strategy: None
  n_estimators: 100
  n_jobs: -1
  num_parallel_tree: None
  random_state: 42
  reg_alpha: None
  reg_lambda: None
  sampling_method: None
  scale_pos_weight: None
  subsample: 0.8
  tree_method: None
  validate_parameters: None
  verbosity: None

Training model...
Training Complete


In [15]:
#MAKING PREDICTIONS
y_train_pred = model.predict(train_x)
y_test_pred = model.predict(test_x)

In [16]:
train_y

,ClosePrice
2,875000.0
3,875000.0
4,849000.0
15,1100000.0
16,760000.0
...,...
178477,1530963.0
178478,1032000.0
178483,1564380.0
178494,669000.0


In [17]:
# Calculate metrics
def evaluate_model(y_true, y_pred, dataset_name):
    # R² Score
    r2 = r2_score(y_true, y_pred)
    
    # Mean Absolute Percentage Error
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    
    # Median Absolute Percentage Error
    mdape = np.median(np.abs((y_true - y_pred) / y_true)) * 100

    
    print(f"\n{dataset_name} Metrics:")
    print(f"  R²:     {r2:.4f}")
    print(f"  MAPE:   {mape:.2f}%")
    print(f"  MdAPE:  {mdape:.2f}%")
    
    return r2, mape, mdape

# Training metrics
train_r2, train_mape, train_mdape = evaluate_model(
    np.array(train_y).reshape(-1), y_train_pred, "TRAINING SET"
)

# Test metrics
test_r2, test_mape, test_mdape = evaluate_model(
    np.array(test_y).reshape(-1), y_test_pred, "TEST SET"
)

# Check for overfitting
print("OVERFITTING CHECK:")
print(f"  R² difference: {train_r2 - test_r2:.4f}")
if train_r2 - test_r2 > 0.1:
    print("  ⚠️  Model may be overfitting (consider regularization)")
elif train_r2 - test_r2 > 0.05:
    print("  ⚠️  Minor overfitting detected")
else:
    print("  ✓  Model generalizes well")


TRAINING SET Metrics:
  R²:     0.9968
  MAPE:   2.21%
  MdAPE:  1.57%

TEST SET Metrics:
  R²:     0.9932
  MAPE:   2.46%
  MdAPE:  1.65%
OVERFITTING CHECK:
  R² difference: 0.0036
  ✓  Model generalizes well
